## Algoritmo KNN

In [1]:
#Importamos las variables 
import pandas as pd 
import matplotlib.pyplot as plt

In [7]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import ( accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay)

## Cargamos el Dataset

In [8]:
iris = load_iris(as_frame=True)

In [9]:
print(iris.DESCR)

.. _iris_dataset:

Iris plants dataset
--------------------

**Data Set Characteristics:**

:Number of Instances: 150 (50 in each of three classes)
:Number of Attributes: 4 numeric, predictive attributes and the class
:Attribute Information:
    - sepal length in cm
    - sepal width in cm
    - petal length in cm
    - petal width in cm
    - class:
            - Iris-Setosa
            - Iris-Versicolour
            - Iris-Virginica

:Summary Statistics:

============== ==== ==== ======= ===== ====================
                Min  Max   Mean    SD   Class Correlation
============== ==== ==== ======= ===== ====================
sepal length:   4.3  7.9   5.84   0.83    0.7826
sepal width:    2.0  4.4   3.05   0.43   -0.4194
petal length:   1.0  6.9   3.76   1.76    0.9490  (high!)
petal width:    0.1  2.5   1.20   0.76    0.9565  (high!)
============== ==== ==== ======= ===== ====================

:Missing Attribute Values: None
:Class Distribution: 33.3% for each of 3 classes.
:Cr

In [10]:
dfIris = iris.frame
dfIris.head(2)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0


In [11]:
dfIris.tail(2)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
148,6.2,3.4,5.4,2.3,2
149,5.9,3.0,5.1,1.8,2


In [17]:
iris.target_names

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

In [22]:
#Cambiamos la columna de Targer por Especie
dfIris.rename(columns={"target" : "species"}, inplace=True)

In [23]:
#Reemplazamos los valores numéricos por las especies
dfIris["species"] = dfIris["species"].map(
    {
        0: "setosa",
        1: "versicolor",
        2: "virginica"
    }
)

In [24]:
dfIris.head(5)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [25]:
dfIris.tail(5)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica
149,5.9,3.0,5.1,1.8,virginica


## Variables predictoras y Variable objetivo

X (Variables predictoras) son aquellas que me permiten hacer la predicción 

y (objetivo) Lo que quiero predecir

In [26]:
#Variables predictoras
X = dfIris.drop(columns=["species"])

In [27]:
#Variable objetivo
y = dfIris["species"]

In [28]:
print("----Variables predictoras----")
print(X.columns)
print("----Variable Objetivo-----")
print(y.name)

----Variables predictoras----
Index(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)'],
      dtype='object')
----Variable Objetivo-----
species


## División del conjunto de datos
*Training Set:*  Datos para entrenamiento.

*Testing Set:* Datos no vistos por el modelo, para prueba (20%).

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X, #Variables predictoras
    y, #Variable objetivo
    test_size = 0.20,
    random_state=42,
    stratify=y
)

In [30]:
print("X_train: ", X_train.shape)
print("X_test: ", X_test.shape)
print("-----------------------")
print("y_train: ", y_train.shape)
print("y_test: ", y_test.shape)

X_train:  (120, 4)
X_test:  (30, 4)
-----------------------
y_train:  (120,)
y_test:  (30,)


## Escalado de las variables

In [31]:
scaler = StandardScaler()

In [32]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [34]:
print("------------Datos sin escalar------------")
print(X_train.head(5))
print("--------Escalados--------")
print(X_train_scaled[:5])

------------Datos sin escalar------------
     sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
8                  4.4               2.9                1.4               0.2
106                4.9               2.5                4.5               1.7
76                 6.8               2.8                4.8               1.4
9                  4.9               3.1                1.5               0.1
89                 5.5               2.5                4.0               1.3
--------Escalados--------
[[-1.72156775 -0.33210111 -1.34572231 -1.32327558]
 [-1.12449223 -1.22765467  0.41450518  0.6517626 ]
 [ 1.14439475 -0.5559895   0.58484978  0.25675496]
 [-1.12449223  0.11567567 -1.28894078 -1.45494479]
 [-0.40800161 -1.22765467  0.13059752  0.12508575]]


## Entrenamiento del modelo 

In [35]:
#Creamos una instancia del Algoritmo 
modelo = KNeighborsClassifier(
    n_neighbors = 3
)

In [36]:
#Entrenamos con los datos de entrenamiento
modelo.fit(
    X_train_scaled,
    y_train
)

KNeighborsClassifier(n_neighbors=3)

## Predicción con los datos de prueba

In [38]:
prediccion = modelo.predict(X_test_scaled)

In [39]:
pd.DataFrame({
    "Especie real" : y_test.values,
    "Especie predicha" : prediccion
}).head(10)

,Especie real,Especie predicha
0,setosa,setosa
1,virginica,virginica
2,versicolor,versicolor
3,versicolor,versicolor
4,setosa,setosa
5,versicolor,versicolor
6,setosa,setosa
7,setosa,setosa
8,virginica,virginica
9,versicolor,versicolor


## Variables inportantes utilizadas por el modelo

## Evaluación del modelo